# Synthesis

## Diagnosing learning problems before trying interventions

This closing notebook synthesizes the workshop investigation. We will not run new training here. The evidence comes from the shared baseline report and the group intervention notebooks.

By the end, you should be able to:

1. summarize a group investigation as evidence, intervention, result, and conclusion;
2. restate the learning problem in terms of data, hypothesis space, objective/optimizer, and evaluation;
3. explain why a diagnosis should name a specific failure mode, not only a broad category;
4. use validation evidence to choose an intervention and test evidence only as a final check;
5. write a defensible diagnostic claim about what the evidence supports or weakens.

The aim is not to rank the groups. The aim is to compare how different explanations were tested.


In [ ]:
# Environment setup. The notebook is designed to run locally and in Colab.
import importlib.util
import os
import subprocess
import sys
import tempfile
from pathlib import Path

os.environ.setdefault(
    "MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "nextgen2026-matplotlib")
)

REPO_URL = "https://github.com/nextgenerationgraduatesprogram/nextgen2026-mlai-workshops.git"
REPO_BRANCH = "workshop3"
PACKAGE_NAME = "nextgen2026_mlai_workshops"

if "google.colab" in sys.modules:
    repo_dir = Path("/content/nextgen2026-mlai-workshops")
    if not repo_dir.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                REPO_BRANCH,
                REPO_URL,
                str(repo_dir),
            ],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_dir)], check=True)
    missing_packages = [
        package_name
        for package_name, module_name in (("pandas", "pandas"), ("torch", "torch"))
        if importlib.util.find_spec(module_name) is None
    ]
    if missing_packages:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_packages], check=True)
    sys.path.insert(0, str(repo_dir / "src"))
else:
    repo_dir = None
    for possible_root in (Path.cwd(), Path.cwd().parent):
        possible_src = possible_root / "src"
        if (possible_src / PACKAGE_NAME).exists():
            repo_dir = possible_root
            sys.path.insert(0, str(possible_src))
            break

for module_name in list(sys.modules):
    if module_name == PACKAGE_NAME or module_name.startswith(f"{PACKAGE_NAME}."):
        del sys.modules[module_name]

print(f"Workshop 3 environment ready. Repository: {repo_dir or 'installed environment'}")


<br>

## 1. Group Report-Back

Use this table to summarize what each group investigated. Record the explanation tested and the evidence used; do not record every run.

| Group | Broad category | Failure mode investigated | Evidence used | Intervention tested | Validation result | Final test check | Conclusion |
|---|---|---|---|---|---|---|---|
| A | Data | | | | | | |
| B | Hypothesis space | | | | | | |
| C | Objective/optimizer | | | | | | |

A final test check is not part of choosing an intervention. It is the last check on the claim after the group has already chosen from validation evidence.


<br>

## 2. Compare the Diagnoses

The useful comparison is mechanism, not rank. Different causes can produce similar symptoms, and a similar intervention can mean different things if it tests a different explanation.

Discuss:

- Which groups saw similar model behaviour but proposed different explanations?
- Which evidence most directly connected a symptom to a cause?
- Which intervention most directly tested its proposed cause?
- Where did the result weaken the first explanation?
- What additional evidence would make the conclusion more defensible?

A metric change is not a conclusion by itself. Say what explanation the result supports or weakens.


<br>

## 3. Components of Learning

The same trained model can be understood through four components of the learning problem.

| Component | Question it asks | Diagnostic role |
|---|---|---|
| Data | What evidence was available? | Checks coverage, measurement, sampling, and match to the intended use. |
| Hypothesis space | What functions were allowed? | Checks whether the model class and inputs can represent the needed relationship. |
| Objective/optimizer | How was a function selected? | Checks whether training dynamics and the objective selected a useful solution. |
| Evaluation | What claim does the evidence support? | Checks where the model works, where it fails, and how strong the claim can be. |

Data, hypothesis space, and objective/optimizer shape the fitted model. Evaluation shapes what we are allowed to claim about that model.


<br>

## 2. A Simple Diagnostic Loop

Do not start by trying random changes. Start by writing a causal story and then test it.

Use the loop:

$$
H_{\mathrm{fail}}
\xrightarrow{E}
H_{\Delta}
\xrightarrow{\Delta}
R
\xrightarrow{J}
C.
$$

Here $E$ denotes evidence, $\Delta$ denotes the intervention, and $J$ denotes the interpretation or judgement step.

| Step | Question |
|---|---|
| Failure hypothesis | What specific failure mode could explain the observed behaviour? |
| Evidence | What plot, table, residual pattern, curve, or slice result supports or weakens that hypothesis? |
| Intervention | What change directly tests that hypothesis? |
| Result | What changed on validation evidence after the intervention? |
| Conclusion | What explanation is now more plausible, less plausible, or unresolved? |

This is the difference between a controlled investigation and a random search.


<br>

## 5. Make the Failure Specific

The more specific the diagnosis, the more meaningful the intervention.

| Broad label | More diagnostic version |
|---|---|
| Data problem | The available evidence may be sparse, mismatched to the intended use, or uneven across operating regimes. Examples include missing combinations and train/deployment mismatch. |
| Model problem | The allowed functions or input representation may not express the relationship the task needs. Examples include capacity, input representation, and activation bias. |
| Training problem | The optimizer, initialization, schedule, or objective may not select a stable or appropriate solution. Examples include scaling/initialization, learning-rate schedule, batch size, epoch budget, and objective choice. |
| Evaluation problem | The metric or slice being inspected may not match the claim we want to make. Examples include relying only on an average metric or using test evidence before a choice is fixed. |

Do not stop at the broad label. Name the concrete failure mode your evidence actually tested.


<br>

## 6. Diagnostic Claim Template

Use this template to explain an ML investigation without turning it into a list of runs.

```text
We diagnosed ______, not just ______.
The evidence was ______.
Our intervention tested the explanation by ______.
The validation result made the explanation more/less plausible because ______.
The final test check ______.
The main limitation of this conclusion is ______.
```

This form forces the conclusion to connect back to evidence rather than to a metric alone.


<br>

## 7. Closing Takeaway

Better ML practice is not trying every lever. It is using evidence to decide which lever answers the question you are actually asking.
